In [ ]:
# kiểm tra runtime

!nvidia-smi
# Still not use gpu so command not found

In [ ]:
# mount Google Drive

from google.colab import drive
drive.mount('/content/drive')
!ls "/content/drive/MyDrive/DATH_CXR"

In [ ]:
# Nhập GitHub token bằng getpass
from getpass import getpass

GITHUB_TOKEN = getpass("GitHub PAT: ")

In [ ]:
# Clone hoặc pull repository từ GitHub
import os
import subprocess
import base64

# Thông tin repository
repo_dir = "/content/DATH-HK261-explainable-cxr"
repo_url = "https://github.com/HuyDuongQuoc/DATH-HK261-explainable-cxr.git"

# Kiểm tra token đã được nhập ở cell trước chưa
if not GITHUB_TOKEN:
    raise ValueError("GitHub token chưa được nhập!")

# Tạo thông tin xác thực
auth = base64.b64encode(
    f"x-access-token:{GITHUB_TOKEN}".encode()
).decode()

git_auth_config = f"http.extraHeader=Authorization: Basic {auth}"

# Clone nếu chưa có repository
if not os.path.exists(repo_dir):

    print("Repository chưa tồn tại. Đang clone...")

    subprocess.run(
        [
            "git",
            "-c",
            git_auth_config,
            "clone",
            repo_url,
            repo_dir
        ],
        check=True
    )

    print("Repository cloned successfully!")

# Pull nếu repository đã tồn tại
else:

    if not os.path.isdir(os.path.join(repo_dir, ".git")):
        raise RuntimeError(
            f"{repo_dir} đã tồn tại nhưng không phải Git repository. "
            "Hãy kiểm tra thư mục trước khi tiếp tục."
        )

    print("Repository đã tồn tại. Đang cập nhật...")

    subprocess.run(
        [
            "git",
            "-C",
            repo_dir,
            "-c",
            git_auth_config,
            "pull"
        ],
        check=True
    )

    print("Repository updated successfully!")

In [ ]:
# Check nội dung repo
%cd /content/DATH-HK261-explainable-cxr
!ls

In [ ]:
# chạy smoke test
!python scripts/smoke_test.py \
    --checkpoint-dir "/content/drive/MyDrive/DATH_CXR/checkpoints/smoke_test" \
    --result-dir "/content/results/smoke_test"

In [ ]:
# Kiểm tra checkpoint đã lưu lên Drive
from pathlib import Path

checkpoint = Path(
    "/content/drive/MyDrive/DATH_CXR/checkpoints/"
    "smoke_test/smoke_test_checkpoint.txt"
)

print("Checkpoint exists:", checkpoint.is_file())

In [ ]:
# Kiểm tra và tải kết quả về máy
from pathlib import Path
from google.colab import files

result = Path("/content/results/smoke_test/metrics.json")

if result.is_file():
    print("Result exists: True")
    print(result.read_text(encoding="utf-8"))
    files.download(str(result))
else:
    print("Result exists: False")